In [1]:
# Install dependencies (run once in terminal with venv activated):
#   pip install torch torchvision tqdm kagglehub nltk matplotlib pandas pillow jupyter
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'tqdm', 'kagglehub', 'nltk', 'matplotlib', 'pandas', 'pillow'])


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


0

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
import pandas as pd
import matplotlib.pyplot as plt
import random
import re
import os
import sys
import csv
import math
from PIL import Image
from collections import Counter
from tqdm import tqdm
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)

# MPS fallback for unsupported ops on Apple Silicon
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Mac-compatible device: MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Using device:', device)

Using device: mps


In [3]:
# ── Local output directory ────────────────────────────────────────────────────
OUTPUT_DIR = './model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Phase 1 checkpoint — your existing epoch 23 decoder
PHASE1_CHECKPOINT = os.path.join(OUTPUT_DIR, 'resnet50_attention_model.pth')

# Phase 2 checkpoint — saved during encoder fine-tuning
MODEL_PATH   = os.path.join(OUTPUT_DIR, 'resnet50_attention_finetuned.pth')
VOCAB_PATH   = os.path.join(OUTPUT_DIR, 'vocab.pt')
FEATURE_PATH = os.path.join(OUTPUT_DIR, 'resnet50_features.pt')  # base cached features

print('Output directory ready:', OUTPUT_DIR)
print('Phase 1 checkpoint exists:', os.path.exists(PHASE1_CHECKPOINT))

Output directory ready: ./model_outputs
Phase 1 checkpoint exists: True


In [4]:
import kagglehub
path = kagglehub.dataset_download('adityajn105/flickr8k')
IMAGE_DIR    = os.path.join(path, 'Images')
CAPTION_FILE = os.path.join(path, 'captions.txt')
print(len(os.listdir(IMAGE_DIR)), 'images found')
print('Caption file exists:', os.path.exists(CAPTION_FILE))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


8091 images found
Caption file exists: True


In [5]:
def tokenize(caption: str):
    """Lowercase, strip noise, keep <start>/<end> as single tokens."""
    caption = caption.lower().strip()
    caption = re.sub(r"[^a-z0-9<>' ]", ' ', caption)
    caption = caption.replace('<start>', ' <start> ').replace('<end>', ' <end> ')
    caption = re.sub(r'\s+', ' ', caption).strip()
    return caption.split()

test = '<start> A dog runs across the field . <end>'
print(tokenize(test))

['<start>', 'a', 'dog', 'runs', 'across', 'the', 'field', '<end>']


In [6]:
captions = {}
with open(CAPTION_FILE, 'r') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) >= 2:
            img_name     = row[0]
            caption_text = '<start> ' + row[1].strip() + ' <end>'
            captions.setdefault(img_name, []).append(caption_text)

print('Total images with captions:', len(captions))

Total images with captions: 8091


In [7]:
word_counter = Counter()
SKIP_TOKENS  = {'<pad>', '<start>', '<end>', '<unk>'}
for caps_list in captions.values():
    for cap in caps_list:
        word_counter.update(t for t in tokenize(cap) if t not in SKIP_TOKENS)

SPECIALS = ['<pad>', '<start>', '<end>', '<unk>']
word2idx = {w: i for i, w in enumerate(SPECIALS)}
idx2word = {i: w for i, w in enumerate(SPECIALS)}

idx      = len(SPECIALS)
MIN_FREQ = 5
for w, c in word_counter.items():
    if c >= MIN_FREQ and w not in word2idx:
        word2idx[w] = idx
        idx2word[idx] = w
        idx += 1

vocab_size = len(word2idx)
PAD_IDX    = word2idx['<pad>']
START_IDX  = word2idx['<start>']
END_IDX    = word2idx['<end>']
UNK_IDX    = word2idx['<unk>']

print(f'Vocabulary size: {vocab_size}')
print(f'  <pad>={PAD_IDX}, <start>={START_IDX}, <end>={END_IDX}, <unk>={UNK_IDX}')
torch.save((word2idx, idx2word), VOCAB_PATH)
print('Vocabulary saved.')

Vocabulary size: 2982
  <pad>=0, <start>=1, <end>=2, <unk>=3
Vocabulary saved.


In [8]:
def caption_to_seq(caption: str):
    return [word2idx.get(w, UNK_IDX) for w in tokenize(caption)]

seq = caption_to_seq('<start> a dog runs <end>')
print('Example seq:', seq)
print('Decoded    :', [idx2word[i] for i in seq])

Example seq: [1, 4, 28, 123, 2]
Decoded    : ['<start>', 'a', 'dog', 'runs', '<end>']


## Dataset, DataLoader & Feature Extraction

In [9]:
all_images = list(captions.keys())
random.seed(42)
random.shuffle(all_images)
split        = int(0.9 * len(all_images))
train_images = all_images[:split]
val_images   = all_images[split:]
print(f'Train: {len(train_images)} | Val: {len(val_images)}')

Train: 7281 | Val: 810


In [10]:
# ── Transforms ───────────────────────────────────────────────────────────────
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Encoder: ResNet50, start fully frozen ────────────────────────────────────
resnet  = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
encoder = nn.Sequential(*list(resnet.children())[:-2])
encoder = encoder.to(device).eval()
for p in encoder.parameters():
    p.requires_grad = False

print('Encoder ready — fully frozen for Phase 1.')

Encoder ready — fully frozen for Phase 1.


In [11]:
# ── Cache base features for all images ───────────────────────────────────────
# These are used throughout Phase 1 and for val throughout Phase 2.
# During Phase 2 training epochs, train features are re-extracted on-the-fly
# from the evolving encoder.

if os.path.exists(FEATURE_PATH):
    print('Loading cached features...')
    all_features = torch.load(FEATURE_PATH, map_location='cpu', weights_only=False)
    print(f'Loaded features for {len(all_features)} images.')
else:
    print('Extracting base features for all images...')
    all_features = {}
    encoder.eval()
    with torch.no_grad():
        for img_name in tqdm(all_images):
            img_path = os.path.join(IMAGE_DIR, img_name)
            try:
                img   = Image.open(img_path).convert('RGB')
                img_t = val_transform(img).unsqueeze(0).to(device)
                feat  = encoder(img_t)
                feat  = feat.squeeze(0).permute(1, 2, 0).reshape(49, 2048).cpu()
                all_features[img_name] = feat
            except Exception as e:
                print(f'  Skipping {img_name}: {e}')
    torch.save(all_features, FEATURE_PATH)
    print(f'Base features saved for {len(all_features)} images.')

Loading cached features...
Loaded features for 8091 images.


In [12]:
from torch.utils.data import Dataset, DataLoader

class Flickr8kDataset(Dataset):
    """
    Two modes:
      feature_cache provided → returns cached features (Phase 1 + val in Phase 2)
      feature_cache=None     → extracts on-the-fly from current encoder (Phase 2 train)
    """
    def __init__(self, image_names, max_len=50, feature_cache=None):
        self.data          = []
        self.feature_cache = feature_cache

        for img_name in image_names:
            if feature_cache is not None and img_name not in feature_cache:
                continue
            img_path = os.path.join(IMAGE_DIR, img_name)
            if feature_cache is None and not os.path.exists(img_path):
                continue
            for cap in captions[img_name]:
                seq = caption_to_seq(cap)
                if len(seq) <= max_len:
                    self.data.append((img_name, seq))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name, seq = self.data[idx]
        if self.feature_cache is not None:
            feat = self.feature_cache[img_name]
        else:
            img   = Image.open(os.path.join(IMAGE_DIR, img_name)).convert('RGB')
            img_t = val_transform(img).unsqueeze(0).to(device)
            encoder.eval()
            with torch.no_grad():
                feat = encoder(img_t)
            feat = feat.squeeze(0).permute(1, 2, 0).reshape(49, 2048).cpu()
        return feat, torch.tensor(seq, dtype=torch.long)


def collate_fn(batch):
    feats, seqs = zip(*batch)
    feats   = torch.stack(feats)
    max_len = max(s.size(0) for s in seqs)
    padded  = torch.full((len(seqs), max_len), PAD_IDX, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :s.size(0)] = s
    return feats, padded


def make_loaders(train_cache=None, val_cache=None, batch_size=64):
    """Build fresh loaders — called each epoch in Phase 2 to get updated features."""
    tr = Flickr8kDataset(train_images, feature_cache=train_cache)
    vl = Flickr8kDataset(val_images,   feature_cache=val_cache)
    train_loader = DataLoader(tr, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=0)
    val_loader   = DataLoader(vl, batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=0)
    print(f'Loaders ready — Train: {len(tr)} | Val: {len(vl)}')
    return train_loader, val_loader

# Initial loaders — fully cached (Phase 1)
train_loader, val_loader = make_loaders(
    train_cache=all_features,
    val_cache=all_features,
    batch_size=64
)

Loaders ready — Train: 36405 | Val: 4050


## Model: Attention + LSTM Decoder

In [13]:
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attn_dim):
        super().__init__()
        self.W_enc = nn.Linear(encoder_dim, attn_dim)
        self.W_dec = nn.Linear(decoder_dim, attn_dim)
        self.V     = nn.Linear(attn_dim, 1)

    def forward(self, encoder_out, h):
        e     = self.W_enc(encoder_out)
        d     = self.W_dec(h).unsqueeze(1)
        score = self.V(torch.tanh(e + d))
        alpha = torch.softmax(score, dim=1)
        ctx   = (alpha * encoder_out).sum(1)
        return ctx, alpha.squeeze(-1)


class DecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, encoder_dim=2048,
                 decoder_dim=384, attn_dim=256, dropout=0.6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.attention = Attention(encoder_dim, decoder_dim, attn_dim)
        self.lstm      = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.init_h    = nn.Linear(encoder_dim, decoder_dim)
        self.init_c    = nn.Linear(encoder_dim, decoder_dim)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(decoder_dim, vocab_size)

    def _init_hidden(self, encoder_out):
        mean_enc = encoder_out.mean(dim=1)
        h = torch.tanh(self.init_h(mean_enc))
        c = torch.tanh(self.init_c(mean_enc))
        return h, c

    def forward(self, encoder_out, captions, teacher_forcing_ratio=1.0):
        B, T = captions.size()
        h, c = self._init_hidden(encoder_out)
        outputs     = []
        input_token = captions[:, 0]

        for t in range(T - 1):
            emb     = self.dropout(self.embedding(input_token))
            ctx, _  = self.attention(encoder_out, h)
            lstm_in = self.dropout(torch.cat([emb, ctx], dim=1))
            h, c    = self.lstm(lstm_in, (h, c))
            logits  = self.fc(self.dropout(h))
            outputs.append(logits)

            use_gt      = random.random() < teacher_forcing_ratio
            input_token = captions[:, t + 1] if use_gt else logits.argmax(1)

        return torch.stack(outputs, dim=1)


model = DecoderWithAttention(
    vocab_size   = vocab_size,
    embed_dim    = 256,
    encoder_dim  = 2048,
    decoder_dim  = 384,
    attn_dim     = 256,
    dropout      = 0.6,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Decoder parameters: {total_params:,}')
print('Model ready.')

Decoder parameters: 8,240,295
Model ready.


## Phase 1: Load Checkpoint & Warm Up (Epochs 1–10)

In [14]:
# ── Load your existing epoch 23 decoder checkpoint ───────────────────────────
if os.path.exists(PHASE1_CHECKPOINT):
    ckpt = torch.load(PHASE1_CHECKPOINT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print(f'Loaded decoder checkpoint from epoch {ckpt["epoch"]}.')
else:
    print('WARNING: No Phase 1 checkpoint found — training from scratch.')

# ── Phase 1 optimizer — decoder only ─────────────────────────────────────────
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.15)

optimizer_p1 = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-4)
# Lower LR than before (1e-4 vs 3e-4) — decoder is already trained, just stabilising

scheduler_p1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p1, mode='min', factor=0.5, patience=2
)

print('Phase 1 optimizer ready.')

Loaded decoder checkpoint from epoch 23.
Phase 1 optimizer ready.


In [15]:
def train_epoch(loader, optimizer, tf_ratio, fine_tune_encoder=False):
    model.train()
    if fine_tune_encoder:
        encoder.train()
    else:
        encoder.eval()

    total_loss = 0
    pbar = tqdm(loader, desc='Training', leave=False)
    for feats, caps in pbar:
        feats = feats.to(device)
        caps  = caps.to(device)

        outputs = model(feats, caps, teacher_forcing_ratio=tf_ratio)
        targets = caps[:, 1:]
        loss    = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        if fine_tune_encoder:
            nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / len(loader)


@torch.no_grad()
def val_epoch(loader):
    model.eval()
    encoder.eval()
    total_loss = 0
    for feats, caps in loader:
        feats = feats.to(device)
        caps  = caps.to(device)
        outputs = model(feats, caps, teacher_forcing_ratio=1.0)
        targets = caps[:, 1:]
        loss    = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
        total_loss += loss.item()
    return total_loss / len(loader)

print('Train / val functions ready.')

Train / val functions ready.


In [16]:
# ── Phase 1: Stabilise decoder on cached features (10 epochs) ────────────────
PHASE1_EPOCHS     = 10
best_val_loss_p1  = float('inf')
history_p1        = {'train_loss': [], 'val_loss': []}

print('=== PHASE 1: Decoder warm-up (encoder frozen) ===')

for epoch in range(1, PHASE1_EPOCHS + 1):
    # Keep TF high during warm-up — decoder is re-settling from checkpoint
    tf_ratio = 1.0

    train_loss = train_epoch(train_loader, optimizer_p1, tf_ratio,
                             fine_tune_encoder=False)
    val_loss   = val_epoch(val_loader)
    scheduler_p1.step(val_loss)

    history_p1['train_loss'].append(train_loss)
    history_p1['val_loss'].append(val_loss)

    if val_loss < best_val_loss_p1:
        best_val_loss_p1 = val_loss
        flag = '  ✓'
    else:
        flag = ''

    print(f'P1 Epoch {epoch:2d}/{PHASE1_EPOCHS} | '
          f'Train {train_loss:.4f} | Val {val_loss:.4f}{flag}', flush=True)

print(f'\nPhase 1 complete. Best val loss: {best_val_loss_p1:.4f}')

=== PHASE 1: Decoder warm-up (encoder frozen) ===


P1 Epoch  1/10 | Train 3.8990 | Val 4.0859  ✓


P1 Epoch  2/10 | Train 3.8810 | Val 4.0811  ✓


P1 Epoch  3/10 | Train 3.8686 | Val 4.0831


P1 Epoch  4/10 | Train 3.8561 | Val 4.0816


...

P1 Epoch  6/10 | Train 3.8385 | Val 4.0794


P1 Epoch  7/10 | Train 3.8293 | Val 4.0801


P1 Epoch  8/10 | Train 3.8234 | Val 4.0797


P1 Epoch  9/10 | Train 3.8102 | Val 4.0796


P1 Epoch 10/10 | Train 3.8050 | Val 4.0795

Phase 1 complete. Best val loss: 4.0769


## Phase 2: Encoder Fine-Tuning (Epochs 11+)

In [17]:
# ── Unfreeze encoder layer4 for fine-tuning ──────────────────────────────────
for p in encoder.parameters():
    p.requires_grad = False
for p in encoder[7].parameters():   # layer4 = index 7
    p.requires_grad = True

encoder_params = [p for p in encoder[7].parameters() if p.requires_grad]
layer4_params  = sum(p.numel() for p in encoder_params)
print(f'Encoder layer4 unfrozen — {layer4_params:,} trainable parameters.')

# ── Phase 2 optimizer — two LR groups ────────────────────────────────────────
# Decoder: same LR as Phase 1 end
# Encoder layer4: very small LR to avoid destroying pretrained weights
optimizer_p2 = optim.AdamW([
    {'params': model.parameters(),  'lr': 1e-4},
    {'params': encoder_params,       'lr': 1e-5},
], weight_decay=5e-4)

scheduler_p2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p2, mode='min', factor=0.5, patience=3
)

print('Phase 2 optimizer ready (decoder 1e-4, encoder layer4 1e-5).')

Encoder layer4 unfrozen — 14,964,736 trainable parameters.
Phase 2 optimizer ready (decoder 1e-4, encoder layer4 1e-5).


In [18]:
def extract_train_features():
    """Re-extract training features from the current (updated) encoder.
    Called at the start of each Phase 2 epoch so features stay in sync
    with the evolving encoder weights.
    """
    encoder.eval()
    fresh_features = {}
    with torch.no_grad():
        for img_name in tqdm(train_images, desc='Re-extracting train features', leave=False):
            img_path = os.path.join(IMAGE_DIR, img_name)
            try:
                img   = Image.open(img_path).convert('RGB')
                img_t = val_transform(img).unsqueeze(0).to(device)
                feat  = encoder(img_t)
                feat  = feat.squeeze(0).permute(1, 2, 0).reshape(49, 2048).cpu()
                fresh_features[img_name] = feat
            except Exception as e:
                pass
    return fresh_features

print('Feature re-extraction function ready.')

Feature re-extraction function ready.


In [19]:
# ── Phase 2: Encoder fine-tuning ─────────────────────────────────────────────
PHASE2_EPOCHS     = 30
best_val_loss_p2  = float('inf')
history_p2        = {'train_loss': [], 'val_loss': []}
patience          = 6
epochs_no_improve = 0

# Val features stay fixed (from original frozen encoder) — fair evaluation
val_cache = all_features

print('=== PHASE 2: Encoder fine-tuning (layer4 unfrozen) ===')
print('Note: train features re-extracted each epoch (~3-5 mins per epoch on M2)')
print()

for epoch in range(1, PHASE2_EPOCHS + 1):
    global_epoch = PHASE1_EPOCHS + epoch

    # TF continues decaying from where Phase 1 left off
    tf_ratio = max(0.7, 1.0 - 0.01 * (global_epoch - 10))

    # Re-extract train features from updated encoder each epoch
    print(f'P2 Epoch {epoch:2d}/{PHASE2_EPOCHS} — re-extracting train features...', flush=True)
    train_cache = extract_train_features()

    # Build fresh loaders with updated train features
    train_loader_p2, val_loader_p2 = make_loaders(
        train_cache=train_cache,
        val_cache=val_cache,
        batch_size=32   # smaller batch — on-the-fly was slower, but cache is fine
    )

    train_loss = train_epoch(train_loader_p2, optimizer_p2, tf_ratio,
                             fine_tune_encoder=True)
    val_loss   = val_epoch(val_loader_p2)
    scheduler_p2.step(val_loss)

    history_p2['train_loss'].append(train_loss)
    history_p2['val_loss'].append(val_loss)

    if val_loss < best_val_loss_p2:
        best_val_loss_p2  = val_loss
        epochs_no_improve = 0
        torch.save({
            'epoch':     global_epoch,
            'model':     model.state_dict(),
            'encoder':   encoder.state_dict(),
            'optimizer': optimizer_p2.state_dict(),
            'word2idx':  word2idx,
            'idx2word':  idx2word,
        }, MODEL_PATH)
        flag = '  ✓ saved'
    else:
        epochs_no_improve += 1
        flag = f'  (no improve {epochs_no_improve}/{patience})'

    print(f'P2 Epoch {epoch:2d}/{PHASE2_EPOCHS} | '
          f'Train {train_loss:.4f} | Val {val_loss:.4f} | '
          f'TF {tf_ratio:.2f}{flag}', flush=True)
    print()

    if epochs_no_improve >= patience:
        print(f'Early stopping triggered after Phase 2 epoch {epoch}.', flush=True)
        break

print(f'\nPhase 2 complete. Best val loss: {best_val_loss_p2:.4f}')

=== PHASE 2: Encoder fine-tuning (layer4 unfrozen) ===
Note: train features re-extracted each epoch (~3-5 mins per epoch on M2)

P2 Epoch  1/30 — re-extracting train features...


Loaders ready — Train: 36405 | Val: 4050


P2 Epoch  1/30 | Train 3.8393 | Val 4.0853 | TF 0.99  ✓ saved

P2 Epoch  2/30 — re-extracting train features...


Loaders ready — Train: 36405 | Val: 4050


P2 Epoch  2/30 | Train 3.8430 | Val 4.0799 | TF 0.98  ✓ saved

P2 Epoch  3/30 — re-extracting train features...


...

P2 Epoch  8/30 | Train 3.8604 | Val 4.0811 | TF 0.92  (no improve 4/6)

P2 Epoch  9/30 — re-extracting train features...


Loaders ready — Train: 36405 | Val: 4050


P2 Epoch  9/30 | Train 3.8625 | Val 4.0799 | TF 0.91  (no improve 5/6)

P2 Epoch 10/30 — re-extracting train features...


Loaders ready — Train: 36405 | Val: 4050


P2 Epoch 10/30 | Train 3.8628 | Val 4.0825 | TF 0.90  (no improve 6/6)

Early stopping triggered after Phase 2 epoch 10.

Phase 2 complete. Best val loss: 4.0759


In [ ]:
# ── Combined loss plot across both phases ────────────────────────────────────
all_train = history_p1['train_loss'] + history_p2['train_loss']
all_val   = history_p1['val_loss']   + history_p2['val_loss']
epochs    = list(range(1, len(all_train) + 1))

plt.figure(figsize=(12, 6))
plt.plot(epochs, all_train, label='Train Loss', marker='o', linewidth=2)
plt.plot(epochs, all_val,   label='Val Loss',   marker='s', linewidth=2)
plt.axvline(x=PHASE1_EPOCHS + 0.5, color='gray', linestyle='--',
            label='Encoder unfrozen (Phase 2 start)')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss',  fontsize=12)
plt.title('Training and Validation Loss — Staged Fine-Tuning', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plot_path = os.path.join(OUTPUT_DIR, 'loss_curves_finetuned.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f'Plot saved to: {plot_path}')
plt.show()

## BLEU-4 Evaluation

In [20]:
@torch.no_grad()
def beam_search(feature, beam_size=5, max_len=50):
    """
    feature : (49, 2048) CPU tensor for a single image.
    Returns : list of token strings (best caption, without <start>/<end>).
    """
    model.eval()
    enc  = feature.unsqueeze(0).to(device)
    h, c = model._init_hidden(enc)

    beams     = [([START_IDX], 0.0, h.clone(), c.clone())]
    completed = []

    for _ in range(max_len):
        new_beams = []
        for seq, score, h, c in beams:
            tok     = torch.tensor([seq[-1]], device=device)
            emb     = model.embedding(tok)
            ctx, _  = model.attention(enc, h)
            lstm_in = torch.cat([emb, ctx], dim=1)
            h_new, c_new = model.lstm(lstm_in, (h, c))
            logits  = model.fc(model.dropout(h_new))

            logits[0, PAD_IDX] = -1e9
            log_probs           = torch.log_softmax(logits, dim=-1)
            topk_lp, topk_idx   = log_probs[0].topk(beam_size)

            for lp, idx in zip(topk_lp.tolist(), topk_idx.tolist()):
                new_seq   = seq + [idx]
                new_score = score + lp
                if idx == END_IDX:
                    completed.append((new_seq, new_score))
                else:
                    new_beams.append((new_seq, new_score, h_new, c_new))

        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:beam_size]
        if not beams:
            break

    if not completed:
        completed = [(b[0], b[1]) for b in beams]

    best_seq, _ = max(completed, key=lambda x: x[1] / len(x[0]))
    tokens = [idx2word[i] for i in best_seq if i not in (START_IDX, END_IDX, PAD_IDX)]
    return tokens

print('Beam search ready.')

Beam search ready.


In [22]:
# ── Load best Phase 2 checkpoint ─────────────────────────────────────────────
ckpt = torch.load('./model_outputs/resnet50_attention_model.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
if 'encoder' in ckpt:
    encoder.load_state_dict(ckpt['encoder'])
print(f'Loaded best model from epoch {ckpt["epoch"]}.')

# ── Re-extract val features using fine-tuned encoder ─────────────────────────
print('Re-extracting val features with fine-tuned encoder...')
encoder.eval()
finetuned_val_features = {}
with torch.no_grad():
    for img_name in tqdm(val_images, desc='Val features'):
        img_path = os.path.join(IMAGE_DIR, img_name)
        try:
            img   = Image.open(img_path).convert('RGB')
            img_t = val_transform(img).unsqueeze(0).to(device)
            feat  = encoder(img_t)
            feat  = feat.squeeze(0).permute(1, 2, 0).reshape(49, 2048).cpu()
            finetuned_val_features[img_name] = feat
        except Exception as e:
            pass

# ── BLEU-4 ────────────────────────────────────────────────────────────────────
smoothie  = SmoothingFunction().method4
refs_all, hyps_all = [], []

for img_name in tqdm(val_images, desc='BLEU eval'):
    if img_name not in finetuned_val_features:
        continue
    feat = finetuned_val_features[img_name]
    hyp  = beam_search(feat, beam_size=7)
    refs = [tokenize(c) for c in captions[img_name]]
    hyps_all.append(hyp)
    refs_all.append(refs)

bleu4 = corpus_bleu(refs_all, hyps_all, smoothing_function=smoothie)
print(f'\nValidation BLEU-4 (fine-tuned encoder): {bleu4:.4f}')

Loaded best model from epoch 23.
Re-extracting val features with fine-tuned encoder...


BLEU eval: 100%|██████████| 810/810 [10:44<00:00,  1.26it/s]


Validation BLEU-4 (fine-tuned encoder): 0.2152


## Generate a Caption for Your Own Image

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
if 'encoder' in ckpt:
    encoder.load_state_dict(ckpt['encoder'])
model.eval()
encoder.eval()
print(f'Loaded best model from epoch {ckpt["epoch"]}.')


@torch.no_grad()
def generate_caption(image_path: str, beam_size: int = 5):
    """Load a local image file and return a generated caption string."""
    img   = Image.open(image_path).convert('RGB')
    img_t = val_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = encoder(img_t)
    feat = feat.squeeze(0).permute(1, 2, 0).reshape(49, 2048).cpu()

    tokens = beam_search(feat, beam_size=beam_size)
    return ' '.join(tokens)


# ── Point this to any image on your machine ───────────────────────────────────
image_path = '/path/to/your/image.jpg'   # <-- change this

if os.path.exists(image_path):
    caption = generate_caption(image_path)
    print(f'\n🖼️  {image_path}')
    print(f'Caption: {caption}')
else:
    print(f'Image not found: {image_path}')
    print('Update image_path above to point to a real image file.')